In [0]:
import json, os
from pyspark.sql import functions as F, Window
CATALOG= "hawkerreach"

In [0]:
# all model values come from assumptions.json at the repo root (shared with the app); pull the Git folder before each run
if not os.path.exists("assumptions.json"):
    raise FileNotFoundError(f"assumptions.json not found in {os.getcwd()}: run this notebook from a Git folder clone of the repo")
with open("assumptions.json") as f:
    A= {k: v["value"] for k, v in json.load(f).items()}

EATING_SHARE= A["diners.eating_share"]              # 0.83, NEA Perception Survey 2018
VISITS_BASE= A["diners.visits_per_week.base"]       # base case 3
UNDERCOUNT_FACTOR= A["undercount_factor"]           # share of median residents per stall
DINERS_PER_RESIDENT= EATING_SHARE * VISITS_BASE / 7
print(f"eating share {EATING_SHARE} | visits/week {VISITS_BASE} | diners per resident per day {DINERS_PER_RESIDENT:.4f} | undercount factor {UNDERCOUNT_FACTOR}")

In [0]:
# every centre within 2 km of each block, ranked; ties broken by centre_id, same rule as 04_gold_access
w= Window.partitionBy("block_id").orderBy("walk_m", "centre_id")
ranked= spark.table(f"{CATALOG}.silver.dist_matrix").withColumn("rank", F.row_number().over(w))

blocks= (spark.table(f"{CATALOG}.silver.blocks")
         .select("block_id", "lon", "lat", F.col("subzone_c").alias("subzone_id"), "est_residents", "est_seniors"))
options= (blocks.join(ranked, "block_id", "left")  # a block with no centre within 2 km keeps one row with centre_id null
    .withColumn("diners_per_day", F.col("est_residents") * DINERS_PER_RESIDENT)
    .select("block_id", "centre_id", "walk_m", "rank", "lon", "lat", "subzone_id", "est_residents", "est_seniors", "diners_per_day"))

options.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.block_options")

In [0]:
o= spark.table(f"{CATALOG}.gold.block_options")
n_blocks= spark.table(f"{CATALOG}.silver.blocks").count()
no_centre= o.filter(F.col("centre_id").isNull())
print("rows:", o.count(), "| blocks:", o.select("block_id").distinct().count(), "of", n_blocks, "| blocks with no centre within 2 km:", no_centre.count())

assert o.select("block_id").distinct().count() == n_blocks, "a block is missing from block_options"
assert no_centre.filter(F.col("rank").isNotNull()).count() == 0, "a no-centre row has a rank"
assert o.filter(F.col("centre_id").isNotNull() & F.col("rank").isNull()).count() == 0, "a centre row has no rank"
gaps= o.filter(F.col("centre_id").isNotNull()).groupBy("block_id").agg(F.count("*").alias("n"), F.max("rank").alias("top")).filter("n != top")
assert gaps.count() == 0, "ranks have gaps"

# ranks 1-3 must match 04_gold_access exactly (needs block_access v2 from Spec 1)
ba= spark.table(f"{CATALOG}.gold.block_access")
if "second_centre_id" in ba.columns:
    mine= (o.groupBy("block_id")
            .agg(*[F.max(F.when(F.col("rank") == r, F.col("centre_id"))).alias(c)
                   for r, c in [(1, "nearest_centre_id"), (2, "second_centre_id"), (3, "third_centre_id")]]))
    cols= ["nearest_centre_id", "second_centre_id", "third_centre_id"]
    diff= mine.alias("m").join(ba.alias("b"), "block_id").filter(" OR ".join(f"NOT (m.{c} <=> b.{c})" for c in cols))
    assert diff.count() == 0, f"{diff.count()} blocks rank their top 3 differently from block_access"
    print("ranks 1-3 match block_access")
else:
    print("skipped block_access check: block_access v2 (second_centre_id) not built yet")

In [0]:
# baseline = residents whose nearest centre it is
centres= spark.table(f"{CATALOG}.silver.centres").select("centre_id", "cooked_food_stalls")
baseline= o.filter("rank = 1").groupBy("centre_id").agg(F.sum("est_residents").alias("baseline_residents"))

demand= (centres.join(baseline, "centre_id", "left")
    .fillna(0, ["baseline_residents"])  # never the nearest centre to any block
    .withColumn("baseline_diners_per_day", F.col("baseline_residents") * DINERS_PER_RESIDENT)
    .withColumn("residents_per_stall", F.col("baseline_residents") / F.col("cooked_food_stalls")))
median= demand.agg(F.expr("percentile(residents_per_stall, 0.5)")).first()[0]
demand= (demand.withColumn("possibly_undercounted", F.col("residents_per_stall") < UNDERCOUNT_FACTOR * median)
               .select("centre_id", "baseline_residents", "baseline_diners_per_day", "residents_per_stall", "possibly_undercounted"))

demand.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.centre_demand")

In [0]:
d= spark.table(f"{CATALOG}.gold.centre_demand")
served= o.filter("rank = 1").agg(F.sum("est_residents")).first()[0]
print("centres:", d.count(), "| median residents per stall:", round(median), "| baseline residents:", round(d.agg(F.sum("baseline_residents")).first()[0]), "of", round(served))
assert abs(d.agg(F.sum("baseline_residents")).first()[0] - served) < 0.01, "baseline residents don't add up to residents with a centre within 2 km"

# expect Amoy Street, Market Street, Newton, Adam Road, East Coast Lagoon, Pasir Panjang among these
display(d.filter("possibly_undercounted")
         .join(spark.table(f"{CATALOG}.silver.centres").select("centre_id", "name", "cooked_food_stalls"), "centre_id")
         .orderBy("residents_per_stall"))